# Homework 3: Supervised machine learning

UIC CS 418, Fall 2024

_According to the **Academic Integrity Policy** of this course, all work submitted for grading must be done individually, unless otherwise specified. While we encourage you to talk to your peers and learn from them, this interaction must be superficial with regards to all work submitted for grading. This means you cannot work in teams, you cannot work side-by-side, you cannot submit someone else’s work (partial or complete) as your own. In particular, note that you are guilty of academic dishonesty if you extend or receive any kind of unauthorized assistance. Absolutely no transfer of program code between students is permitted (paper or electronic), and you may not solicit code from family, friends, or online forums. Other examples of academic dishonesty include emailing your program to another student, copying-pasting code from the internet, working in a group on a homework assignment, and allowing a tutor, TA, or another individual to write an answer for you. Academic dishonesty is unacceptable, and penalties range from failure to expulsion from the university; cases are handled via the official student conduct process described at https://dos.uic.edu/conductforstudents.shtml._

This homework is an individual assignment for all graduate students. Undergraduate students are allowed to work in pairs and submit one homework assignment per pair. There will be no extra credit given to undergraduate students who choose to work alone. The pairs of students who choose to work together and submit one homework assignment together still need to abide by the Academic Integrity Policy and not share or receive help from others (except each other).


## Due Date

This assignment is due at 11:59pm Monday, November 16th, 2024. 


### What to Submit

You need to complete all code and answer all questions denoted by **Q#** (each one is under a bike image) in this notebook. When you are done, you should export **`hw3.ipynb`** with your answers as a PDF file, upload that file `hw3.pdf` to *Homework 3 - Written Part* on Gradescope, tagging each question. 

You need to copy all functions that are part of questions Q1-Q9 to `hw3.py`. That includes `process()`, `process_all()`, `create_features()`, `create_labels()`, `class MajorityLabelClassifier()`, `learn_classifier()`, `evaluate_classifier()`, `best_model_selection()` and `classify_tweets()`. You need to upload a completed Jupyter notebook (`hw3.ipynb` file) and `hw3.py` to *Homework 3 - code* on Gradescope. To help you get started, we have provided a template file (`hw3_template.py`) containing imports, some hints, and function skeletons.



For undergraduate students who work in a team of two, only one student needs to submit the homework and just tag the other student on Gradescope.

#### Autograding

Questions will be graded based on both manual grading and an Autograder which will run on your `hw3.py` file. 
This assignment is graded on the basis of correctness and 70/100 points are given by the autograder. The remaining 30 points will be manually graded (10 points for Q8, 2 points for Q9 and 8 points for correctly running everything in the Jupyter notebook). 

Most of the questions are graded independently. This means that if you have an error in a question, it will not be propagated to another question. However, the final question Q9 will check your overall pipeline and is rather expensive to run on Gradescope. Therefore, you should disable its auto-grading on Gradescope until you have implemented and passed Q1 to Q7. A function `test_pipeline()` is provided in the hw3_template.py file that returns `False` by default to disable auto-grading of Q9. Once you complete the implementation of Q1 to Q7, you can enable auto-grading of the whole pipeline by setting `test_pipeline()` to return `True`.”

The test cases will take a bit longer to execute. Make use of the resources wisely by first testing your functions in your notebook or making local test cases.

In [1]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import nltk #install using "conda install anaconda::nltk"
import sklearn 
import string
import re # helps you filter urls
from IPython.display import display, Latex, Markdown

# Classifying tweets [100%]

In this problem, you will be analyzing Twitter data extracted using [the Twitter API](https://dev.twitter.com/overview/api). We have provided  the data containing tweets posted by the following six X/Twitter accounts: `realDonaldTrump, JDVance, GOP, KamalaHarris, Tim_Walz, TheDemocrats` in `train.csv` file.

The  csv file contains the following columns:
- `Content`: type of content
- 'handle' : name of the X account

The tweets have been divided into two parts - train and test available to you in CSV files. For train, both the `handle` and `Content` attributes were provided but for test, `handle` is hidden.

The overarching goal of the problem is to "predict" the political inclination or affiliation (Republican/Democratic) of individual tweets. The ground truth (i.e., true class labels) is determined from the `handle` of the tweet as follows
- `realDonaldTrump, JDVance, GOP` are Republicans
- `KamalaHarris, Tim_Walz, TheDemocrats` are Democrats

Thus, this is a binary classification problem. 



We will The problem proceeds in three stages:
- **Text processing (25%)**: We will clean up the raw tweet text using the various functions offered by the [nltk](http://www.nltk.org/genindex.html) package.
- **Feature construction (25%)**: In this part, we will construct bag-of-words feature vectors and training labels from the processed text of tweets and the `screen_name` columns respectively.
- **Classification (50%)**: Using the features derived, we will use [sklearn](http://scikit-learn.org/stable/modules/classes.html) package to learn a model which classifies the tweets as desired. 

You will use two new python packages in this problem: `nltk` and `sklearn`, both of which should be available with anaconda. However, NLTK comes with many corpora, toy grammars, trained models, etc, which have to be downloaded manually. This assignment requires NLTK's stopwords list, POS tagger, and WordNetLemmatizer. Install them using conda

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')    # Use nltk downloader to download resource "punkt" once
nltk.download('punkt_tab')    # Use nltk downloader to download resource "punkt_tab" once

from nltk import pos_tag
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

# Verify that the following commands work for you, before moving on.

lemmatizer= nltk.stem.wordnet.WordNetLemmatizer()
stopwords= stopwords.words('english')

[nltk_data] Downloading package stopwords to /Users/lyt/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/lyt/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/lyt/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /Users/lyt/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/lyt/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
train = pd.read_csv("train.csv") 
train_tweets = train["Content"]
print(train_tweets.head(),'\n',len(train_tweets))

0    . and I pledge to be trusted partners who will...
1    Great to be with Liza Colón-Zayas to cook up a...
2    Kamala Harris wants to flood the United States...
3    Kamala Harris wants to raise taxes on American...
4    When President Trump is elected he will usher ...
Name: Content, dtype: object 
 1125


In [4]:
#look at tweet 199 to get an idea 

print(train_tweets[199])

emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "]+", flags=re.UNICODE)
print(emoji_pattern.sub(r'', train_tweets[199])) # no emoji

#first emoji, then tokenize.. and so on
print(train_tweets[199])


tokens = word_tokenize(train_tweets[199])
print(tokens,'\n')

Times Square was bejeweled this morning with our new ads 💎
Times Square was bejeweled this morning with our new ads 
Times Square was bejeweled this morning with our new ads 💎
['Times', 'Square', 'was', 'bejeweled', 'this', 'morning', 'with', 'our', 'new', 'ads', '💎'] 



Let's begin!

## A. Text Processing [25%]

You first task to fill in the following function which processes and tokenizes raw text. The generated list of tokens should meet the following specifications:
1. The tokens must all be in lower case.
2. The tokens should appear in the same order as in the raw text.
3. The tokens must be in their lemmatized form. If a word cannot be lemmatized (i.e, you get an exception), simply catch it and ignore it. These words will not appear in the token list.
4. The tokens must not contain any punctuations. Punctuations should be handled as follows: (a) Apostrophe of the form `'s` must be ignored. e.g., `She's` becomes `she`. (b) Other apostrophes should be omitted. e.g, `don't` becomes `dont`. (c) Words must be broken at the hyphen and other punctuations. 
5. The tokens must not contain any part of a url.

Part of your work is to figure out a logical order to carry out the above operations. You may find `string.punctuation` useful, to get hold of all punctuation symbols. Look for [regular expressions](https://docs.python.org/3/library/re.html) capturing urls in the text. Your tokens must be of type `str`. Use `nltk.word_tokenize()` for tokenization once you have handled punctuation in the manner specified above. 

You would want to take a look at the `lemmatize()` function [here](https://www.nltk.org/_modules/nltk/stem/wordnet.html).

In order for `lemmatize()` to give you the root form for any word, you have to provide the context in which you want to lemmatize through the `pos` parameter: `lemmatizer.lemmatize(word, pos=SOMEVALUE)`. The context should be the part of speech (POS) for that word. The good news is you do not have to manually write out the lexical categories for each word because [nltk.pos_tag()](https://www.nltk.org/book/ch05.html) will do this for you. Now you just need to use the results from `pos_tag()` for the `pos` parameter.
However, you can notice the POS tag returned from `pos_tag()` is in different format than the expected pos by `lemmatizer`.

> pos

(Syntactic category): n for noun files, v for verb files, a for adjective files, r for adverb words.

You need to map these pos appropriately. `nltk.help.upenn_tagset()` provides description of each tag returned by `pos_tag()`.

<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br><br>

## Q1 (15%):

In [5]:
#tweet processing example
example_tweet = "🇺🇸 making world's best is what we're proud of doing off-the-shelf! And more proud:https://www.google.com"
#tempstr = gop_tweets[temp_num]
print(example_tweet)

emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "]+", flags=re.UNICODE)

#handle url
print(re.sub(r'http\S+', '', example_tweet))
example_tweet = re.sub(r'http\S+', '', example_tweet)

#handle emojis
print(emoji_pattern.sub(r'', example_tweet)) # no emoji
example_tweet = emoji_pattern.sub(r'', example_tweet)

#handle apostrophe 's
print(example_tweet)
print(example_tweet.replace("'s","")) #replace 's to 
example_tweet = example_tweet.replace("'s","")

#handle other apostrophe
example_tweet = example_tweet.replace("'","")

#handle all other punctuation
print(re.sub(r'[' + string.punctuation + r']+', ' ', example_tweet).strip())
example_tweet = re.sub(r'[' + string.punctuation + r']+', ' ', example_tweet).strip()

#handle lowercase
example_tweet = example_tweet.lower()

tokens = word_tokenize(example_tweet)

print(tokens,'\n')

#lemmatize tokens
#lemmatizer=nltk.stem.wordnet.WordNetLemmatizer()


#lemmatize single word or token
token_lemma = lemmatizer.lemmatize(tokens[0])
print(token_lemma)

#lemmatize a list of workds or tokens
tweet_lemma = [lemmatizer.lemmatize(token) for token in tokens]
print(tweet_lemma)


🇺🇸 making world's best is what we're proud of doing off-the-shelf! And more proud:https://www.google.com
🇺🇸 making world's best is what we're proud of doing off-the-shelf! And more proud:
 making world's best is what we're proud of doing off-the-shelf! And more proud:
 making world's best is what we're proud of doing off-the-shelf! And more proud:
 making world best is what we're proud of doing off-the-shelf! And more proud:
making world best is what were proud of doing off the shelf  And more proud
['making', 'world', 'best', 'is', 'what', 'were', 'proud', 'of', 'doing', 'off', 'the', 'shelf', 'and', 'more', 'proud'] 

making
['making', 'world', 'best', 'is', 'what', 'were', 'proud', 'of', 'doing', 'off', 'the', 'shelf', 'and', 'more', 'proud']


In [6]:
# Convert part of speech tag from nltk.pos_tag to word net compatible format
# Simple mapping based on first letter of return tag to make grading consistent
# Everything else will be considered noun 'n'

# 14% credits
def get_wordnet_pos(treebank_tag): #no need to change this function - used to tag tokens for context specification
    if treebank_tag.startswith('J'):
        return nltk.corpus.wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return nltk.corpus.wordnet.VERB
    elif treebank_tag.startswith('R'):
        return nltk.corpus.wordnet.ADV
    else:
        return nltk.corpus.wordnet.NOUN
    
def process(text, lemmatizer=nltk.stem.wordnet.WordNetLemmatizer()):
    """ Normalizes case and handles punctuation
    Inputs:
        text: str: raw text
        lemmatizer: an instance of a class implementing the lemmatize() method
                    (the default argument is of type nltk.stem.wordnet.WordNetLemmatizer)
    Outputs:
        list(str): tokenized text
    """
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "]+", flags=re.UNICODE)

    #handle url
    #print(re.sub(r'http\S+', '', text))
    text = re.sub(r'http\S+', '', text)

    #handle emojis
    #print(emoji_pattern.sub(r'', text)) # no emoji
    text = emoji_pattern.sub(r'', text)

    #handle apostrophe 's
    #print(text)
    #print(text.replace("'s","")) #replace 's to 
    text = text.replace("'s","")

    #handle other apostrophe
    text = text.replace("'","")

    #handle all other punctuation
    #print(re.sub(r'[' + string.punctuation + r']+', ' ', text).strip())
    #text = re.sub(r'[' + string.punctuation + r']+', ' ', text).strip()

    #have to handle unicode characters
    #text = re.sub(r'[^\x00-\x7F]+','', text)

    #handle all other punctuation

    text = re.sub(r'[^\w\s]', ' ', text)
    
    #lower case
    text = text.lower()

    # get tokens and tag them
    tokens = word_tokenize(text)
    tagged_tokens = pos_tag(tokens)

    #print(tokens,'\n')

    #lemmatize tokens
    lemmatizer=nltk.stem.wordnet.WordNetLemmatizer()
    

    #lemmatize single word or token
    #token_lemma = lemmatizer.lemmatize(tokens[0])
    #print(tweet_lemma)

    #lemmatize a list of workds or tokens
    

    text_lemma = [lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token, tag in tagged_tokens]
    #print(tweet_lemma)
    return text_lemma


In [7]:
print(process(train_tweets[1])) #this case needs to be handled done

['great', 'to', 'be', 'with', 'liza', 'colón', 'zayas', 'to', 'cook', 'up', 'an', 'important', 'message']


You can test the above function as follows. Try to make your test strings as exhaustive as possible. Some checks are:

In [8]:
# 1% credit
print(process("I'm doing well! How about you?"))
# ['im', 'do', 'well', 'how', 'about', 'you']

print(process("Education is the ability to listen to almost anything without losing your temper or your self-confidence."))
# ['education', 'be', 'the', 'ability', 'to', 'listen', 'to', 'almost', 'anything', 'without', 'lose', 'your', 'temper', 'or', 'your', 'self', 'confidence']

print(process("been had done languages cities mice"))
# ['be', 'have', 'do', 'language', 'city', 'mice']

print(process("It's hilarious. Check it out http://t.co/dummyurl"))
# ['it', 'hilarious', 'check', 'it', 'out']

print(process("See it Sunday morning at 8:30a on RTV6 and our RTV6 app. http:…"))
# ['see', 'it', 'sunday', 'morning', 'at', '8', '30a', 'on', 'rtv6', 'and', 'our', 'rtv6', 'app']





['im', 'do', 'well', 'how', 'about', 'you']
['education', 'be', 'the', 'ability', 'to', 'listen', 'to', 'almost', 'anything', 'without', 'lose', 'your', 'temper', 'or', 'your', 'self', 'confidence']
['be', 'have', 'do', 'language', 'city', 'mice']
['it', 'hilarious', 'check', 'it', 'out']
['see', 'it', 'sunday', 'morning', 'at', '8', '30a', 'on', 'rtv6', 'and', 'our', 'rtv6', 'app']


<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br><br>

## Q2 (10%):

You will now use the `process()` function we implemented to convert the pandas dataframe we just loaded from tweets_train.csv file. Your function should be able to handle any data frame which contains a column called `Content`. The data frame you return should replace every string in `Content` with the result of `process()` and retain all other columns as such. Do not change the order of rows/columns. Before writing `process_all()`, load the data into a DataFrame and look at its format:

In [9]:
#tweets = pd.read_csv("tweets_train.csv", na_filter=False)
#display(tweets.head())

In [10]:
# 9% credits
def process_all(df, lemmatizer=nltk.stem.wordnet.WordNetLemmatizer()):
    """ process all text in the dataframe using process() function.
    Inputs
        df: pd.DataFrame: dataframe containing a column 'Content' loaded from the CSV file
        lemmatizer: an instance of a class implementing the lemmatize() method
                    (the default argument is of type nltk.stem.wordnet.WordNetLemmatizer)
    Outputs
        pd.DataFrame: dataframe in which the values of Content column have been changed from str to list(str),
                        the output from process() function. Other columns are unaffected.
    """
    tweets = df["Content"].apply(str)
    content = []
    #print(len(tweets))
    for tweet in tweets:
        #print(" ".join(process("See it Sunday morning at 8:30a on RTV6 and our RTV6 app. http:…")))
        
        content.append(process(tweet)) #this is correct
        #content.append(" ".join(process(tweet))) #this is wrong

    df["Content"] = content
    return df

    


In [11]:
# test your code
# 1% credit
processed_tweets = process_all(train)
#print(processed_tweets["Content"])
print(processed_tweets.head())

#                                             Content    handle
#0  [and, i, pledge, to, be, trust, partner, who, ...  Tim_Walz
#1  [great, to, be, with, liza, colón, zayas, to, ...  Tim_Walz
#2  [kamala, harris, want, to, flood, the, united,...   JDVance
#3  [kamala, harris, want, to, raise, tax, on, ame...   JDVance
#4  [when, president, trump, be, elect, he, will, ...       GOP


                                             Content    handle
0  [and, i, pledge, to, be, trust, partner, who, ...  Tim_Walz
1  [great, to, be, with, liza, colón, zayas, to, ...  Tim_Walz
2  [kamala, harris, want, to, flood, the, united,...   JDVance
3  [kamala, harris, want, to, raise, tax, on, ame...   JDVance
4  [when, president, trump, be, elect, he, will, ...       GOP


## B. Feature Construction [25%]

The next step is to derive feature vectors from the tokenized tweets. In this section, you will be constructing a bag-of-words TF-IDF feature vector. But before that, as you may have guessed, the number of possible words is prohibitively large and not all of them may be useful for our classification task. We need to determine which words to retain, and which to omit. A common heuristic is to construct a frequency distribution of words in the corpus and prune out the head and tail of the distribution. The intuition of the above operation is as follows. Very common words (i.e. stopwords) add almost no information regarding similarity of two pieces of text. Similarly with very rare words. NLTK has a list of in-built stop words which is a good substitute for head of the distribution. We will consider a word rare if it occurs only in a single document (row) in whole of `tweets_train.csv`. 

<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br><br>

## Q3 (15%):

Construct a sparse matrix of features for each tweet with the help of `sklearn.feature_extraction.text.TfidfVectorizer` (documentation [here](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)). You need to pass a parameter `min_df=2` to filter out the words occuring only in one document in the whole training set. Remember to ignore the stop words as well. You must leave other optional parameters (e.g., `vocab`, `norm`, etc) at their default values. But you may need to use parameters like `lowercase` and `tokenizer` to handle `processed_tweets` that is a `list` of tokens (not raw text).

In [12]:
# 14% credits
def identity(x):
    return x
def create_features(processed_tweets, stop_words):
    """ creates the feature matrix using the processed tweet text
    Inputs:
        processed_tweets: pd.DataFrame: processed tweets read from train/test csv file, containing the column 'Content'
        stop_words: list(str): stop_words by nltk stopwords (after processing)
    Outputs:
        sklearn.feature_extraction.text.TfidfVectorizer: the TfidfVectorizer object used
            we need this to tranform test tweets in the same way as train tweets
        scipy.sparse.csr.csr_matrix: sparse bag-of-words TF-IDF feature matrix
    """
    tfidf = sklearn.feature_extraction.text.TfidfVectorizer(tokenizer=identity, stop_words=stop_words, lowercase=False, min_df=2)
    feature_matrix = tfidf.fit_transform(processed_tweets)    

    return tfidf, feature_matrix


In [13]:
# execute this code 
# 1% credit
# It is recommended to process stopwords according to our data cleaning rules
processed_stopwords = list(np.concatenate([process(word) for word in stopwords]))
(tfidf, X) = create_features(processed_tweets["Content"],processed_stopwords)
# Ignore warning
tfidf, X
print(X.shape)
# Output (should be similar):
#(TfidfVectorizer(lowercase=False, min_df=2,
#                 stop_words=['i', 'me', 'my', 'myself', 'we', 'our', 'ours',
#                             'ourselves', 'you', 'youre', 'youve', 'youll',
#                             'youd', 'your', 'yours', 'yourself', 'yourselves',
#                             'he', 'him', 'his', 'himself', 'she', 'she', 'her',
#                             'hers', 'herself', 'it', 'it', 'it', 'itself', ...],
#                 tokenizer=<function identity at 0x176bf72e0>),
# <1125x1360 sparse matrix of type '<class 'numpy.float64'>'
# 	with 11622 stored elements in Compressed Sparse Row format>)

(1125, 1358)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['b', 'c', 'e', 'f', 'g', 'h', 'j', 'l', 'n', 'p', 'r', 'u', 'v', 'w'] not in stop_words.
  warnings.warn(


<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br><br>

## Q4 (10%):

Also for each tweet, assign a class label (0 or 1) using its `handle`. Use 0 for realDonaldTrump, mike_pence, GOP and 1 for the rest.

In [14]:
print(processed_tweets["handle"].head())

0    Tim_Walz
1    Tim_Walz
2     JDVance
3     JDVance
4         GOP
Name: handle, dtype: object


In [15]:
handles = processed_tweets["handle"]

labels = []

for handle in handles:
    if handle == "TheDemocrats" or handle == "KamalaHarris" or handle == "Tim_Walz":
        labels.append(1)
    else:
        labels.append(0)

labels = np.asarray(labels)        
print(labels,sum(labels), len(labels)-sum(labels))

[1 1 0 ... 0 1 1] 605 520


In [16]:
# 9% credits
def create_labels(processed_tweets):
    """ creates the class labels from handle
    Inputs:
        processed_tweets: pd.DataFrame: tweets read from train file, containing the column 'handle'
    Outputs:
        numpy.ndarray(int): dense binary numpy array of class labels
    """
    
    handles = processed_tweets["handle"]
    labels = []
    
    for handle in handles:
        if handle == "TheDemocrats" or handle == "KamalaHarris" or handle == "Tim_Walz":
            labels.append(1)
        else:
            labels.append(0)

    labels = np.asarray(labels) 
    
    return labels


In [17]:
# execute this code
# 1% credit
y = create_labels(processed_tweets)
print(y)
# 0        0
# 1        1
# 2        1
# 3        1
# 4        1
#         ..
# 17293    0
# 17294    0
# 17295    0
# 17296    1
# 17297    0
# Name: screen_name, Length: 17298, dtype: int32

[1 1 0 ... 0 1 1]


## C. Classification [50%]

And finally, we are ready to put things together and learn a model for the classification of tweets. The classifier you will be using is [`sklearn.linear_model.LogisticRegression`](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LogisticRegression.html) (Logistic Regression). 

At the heart of  is the concept of Regularization or Penalty, which determines how large/small the parmaeters will be and also zero out parameters if there are dependence in the features matrix. `sklearn`'s Logistic provides four regularization functions: `none`, `l2`, `l1`, `elasticnet` (details [here](https://scikit-learn.org/1.5/modules/linear_model.html#logistic-regression) and [here](https://scikit-learn.org/dev/auto_examples/linear_model/plot_logistic_l1_l2_sparsity.html)).

Through the various functions you implement in this part, you will be able to learn a classifier, score a classifier based on how well it performs, use it for prediction tasks and compare it to a baseline.

Specifically, you will carry out the following tasks (Q5-9) in order:

1. Implement and evaluate a simple baseline classifier MajorityLabelClassifier.
2. Implement the `learn_classifier()` function assuming `penalty` is always one of {`none`, `l2`, `l1`, `elasticnet`}. 
3. Implement the `evaluate_classifier()` function which scores a classifier based on accuracy of a given dataset.
4. Implement `best_model_selection()` to perform cross-validation by calling `learn_classifier()` and `evaluate_classifier()` for different folds and determine which of the four penalties performs the best.
5. Go back to `learn_classifier()` and fill in the best penalty. 

<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br><br>

## Q5 (10%):

To determine whether your classifier is performing well, you need to compare it to a baseline classifier. A baseline is generally a simple or trivial classifier and your classifier should beat the baseline in terms of a performance measure such as accuracy. Implement a classifier called `MajorityLabelClassifier` that always predicts the class equal to **mode** of the labels (i.e., the most frequent label) in training data. Part of the code is done for you. Implement the `fit` and `predict` methods. Initialize your classifier appropriately.

In [18]:
# Skeleton of MajorityLabelClassifier is consistent with other sklearn classifiers
# 8% credits
from collections import Counter

class MajorityLabelClassifier():
    """
    A classifier that predicts the mode of training labels
    """
    def __init__(self):
        """
        Initialize your parameter here
        """
        self.majority_label = None

    def fit(self, X, y):
        """
        Implement fit by taking training data X and their labels y and finding the mode of y
        i.e. store your learned parameter
        """
        label_counts = Counter(y)
        self.majority_label = label_counts.most_common(1)[0][0]

    def predict(self, X):
        """
        Implement to give the mode of training labels as a prediction for each data instance in X
        return labels
        """
        return np.full(X.shape[0], self.majority_label)

# 2% credits
# Report the accuracy of your classifier by comparing the predicted label of each example to its true label
baselineClf = MajorityLabelClassifier()
# Use fit and predict methods to get predictions and compare it with the true labels y
baselineClf.fit(X,y)
predictions = baselineClf.predict(X)
print(predictions[0])
# Evaluate the classifier (e.g., using accuracy)
from sklearn.metrics import accuracy_score
train_accuracy = accuracy_score(y, predictions)
print(train_accuracy)
# print(training accuracy) should give 0.5377777777777778 #basically ndems/total = 605/1125

1
0.5377777777777778


<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br><br>

## Q6 (10%):

Implement the `learn_classifier()` function assuming `penalty` is always one of {`none`, `l1`, `l2`, `elasticnet`}. Stick to default values for any other optional parameters.

In [19]:
# 9% credits
def learn_classifier(X_train, y_train, penalty):
    """ learns a classifier from the input features and labels using the penalty function supplied
    Inputs:
        X_train: scipy.sparse.csr.csr_matrix: sparse matrix of features, output of create_features()
        y_train: numpy.ndarray(int): dense binary vector of class labels, output of create_labels()
        penalty: str: penalty function to be used with classifier. [none|l2|l1|elasticnet]
    Outputs:
        sklearn.linear_model.LogisticRegression: classifier learnt from data
    """
    
    clf = sklearn.linear_model.LogisticRegression(
        penalty=penalty,
        solver='saga', # can handle all the possible penalty
        random_state=42,
        max_iter=1000
    )

    # Elasticnet requires an l1_ratio parameter otherwise error
    if penalty == 'elasticnet':
        clf.l1_ratio = 0.5

    clf.fit(X_train, y_train) # learning from the input params

    return clf

In [20]:
# execute code
# 1% credit
classifier = learn_classifier(X, y, 'l2')


<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br><br>

## Q7 (10%):

Now that we know how to learn a classifier, the next step is to evaluate it, ie., characterize how good its classification performance is. This step is necessary to select the best model among a given set of models, or even tune hyperparameters for a given model.

There are two questions that should now come to your mind:
1. **What data to use?** 
    - **Validation Data**: The data used to evaluate a classifier is called **validation data** (or hold-out data), and it is usually different from the data used for training. The model or hyperparameter with the best performance in the held out data is chosen. This approach is relatively fast and simple but vulnerable to biases found in validation set.
    - **Cross-validation**: This approach divides the dataset in $k$ groups (so, called k-fold cross-validation). One of group is used as test set for evaluation and other groups as training set. The model or hyperparameter with the best average performance across all k folds is chosen. For this question you will perform 4-fold cross validation to determine the best penalty. We will keep all other hyperparameters default for now. This approach provides robustness toward biasness in validation set. However, it takes more time.
    
2. **And what metric?** There are several evaluation measures available in the literature (e.g., accuracy, precision, recall, F-1,etc) and different fields have different preferences for specific metrics due to different goals. We will go with accuracy. According to wiki, **accuracy** of a classifier measures the fraction of all data points that are correctly classified by it; it is the ratio of the number of correct classifications to the total number of (correct and incorrect) classifications. `sklearn.metrics` provides a number of performance metrics.

Now, implement the following function.

In [21]:
# 9% credits
def evaluate_classifier(classifier, X_validation, y_validation):
    """ evaluates a classifier based on a supplied validation data
    Inputs:
        classifier: sklearn.linear_model.LogisticRegression: classifer to evaluate
        X_validation: scipy.sparse.csr.csr_matrix: sparse matrix of features
        y_validation: numpy.ndarray(int): dense binary vector of class labels
    Outputs:
        double: accuracy of classifier on the validation data
    """
    acc = classifier.score(X_validation,y_validation)
    return acc

In [22]:
# test your code by evaluating the accuracy on the training data
# 1% credit
accuracy = evaluate_classifier(classifier, X, y)
print(accuracy) 
# should give around 0.9182222222222223 

0.9208888888888889


<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br><br>

## Q8 (10%):

Now it is time to decide which penalty works best by using the cross-validation technique. Write code to split the training data into 4-folds (75% training and 25% validation) by shuffling randomly. For each penalty, record the average accuracy for all folds and determine the best classifier. Since our dataset is balanced (both classes are in almost equal propertion), `sklearn.model_selection.KFold` [doc](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) can be used for cross-validation.

In [23]:
kf = sklearn.model_selection.KFold(n_splits=4, random_state=1, shuffle=True)
kf



KFold(n_splits=4, random_state=1, shuffle=True)

Then use the following code to determine which classifier is the best. 

In [24]:
# 10% credits
def best_model_selection(kf, X, y):
    """
    Select the penalty giving best results using k-fold cross-validation.
    Other parameters should be left default.
    Input:
    kf (sklearn.model_selection.KFold): kf object defined above
    X (scipy.sparse.csr.csr_matrix): training data
    y (array(int)): training labels
    Return:
    best_penalty (string)
    """
    penalties = [None, 'l2', 'l1', 'elasticnet']
    
    average_accuracies = []
    all_accs = []

    for penalty in penalties:
        kth_accuracies = []
        
        # Use the documentation of KFold cross-validation to split ..
        for train_i, test_i in kf.split(X):
            
            X_train, y_train = X[train_i], y[train_i]
            X_test, y_test = X[test_i], y[test_i]
            
            # call learn_classifer() using training split of kth fold
            model = learn_classifier(X_train, y_train, penalty)

            # evaluate on the test split of kth fold
            kth_accuracies.append(evaluate_classifier(model, X_test, y_test))
            all_accs.append(evaluate_classifier(model, X_test, y_test))

        # record avg accuracies and determine best model (penalty)
        average_accuracies.append(np.mean(kth_accuracies))
        
    
    best_penalty = penalties[np.argmax(average_accuracies)]
        # Use the documentation of KFold cross-validation to split ..
        # training data and test data from create_features() and create_labels()
        # call learn_classifer() using training split of kth fold
        # evaluate on the test split of kth fold
        # record avg accuracies and determine best model (penalty)
    return best_penalty, all_accs, average_accuracies

#Test your code
best_penalty, all_accs, average_accuracies = best_model_selection(kf, X, y)
best_penalty

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


'l2'

In [25]:
print(*all_accs,sep='\n')


0.7836879432624113
0.8291814946619217
0.7793594306049823
0.8149466192170819
0.8014184397163121
0.8220640569395018
0.8540925266903915
0.8327402135231317
0.7446808510638298
0.8042704626334519
0.8291814946619217
0.7758007117437722
0.7588652482269503
0.8042704626334519
0.8291814946619217
0.8149466192170819


In [26]:
print(*average_accuracies,sep='\n')

0.8017938719365993
0.8275788092173342
0.7884833800257439
0.8018159561848515


<img src="bikeshare.png" width="100px" align="left" float="left"/>
<br><br>

## Q9 (10%)

We're almost done! It's time to write a nice little wrapper function that will use our model to classify unlabeled tweets from tweets_test.csv file. 

In [27]:
# 8% credits
def classify_tweets(tfidf, classifier, unlabeled_tweets):
    """ predicts class labels for raw tweet text
    Inputs:
        tfidf: sklearn.feature_extraction.text.TfidfVectorizer: the TfidfVectorizer object used on training data
        classifier: sklearn.linear_model.LogisticRegression: classifier learned
        unlabeled_tweets: pd.DataFrame: tweets read from tweets_test.csv
    Outputs:
        numpy.ndarray(int): dense binary vector of class labels for unlabeled tweets
    """
    tweets = process_all(unlabeled_tweets) #process test tweets
    tweets = tweets['Content'] #get the content of the tweets alone after processing

    
    matrix = tfidf.transform(tweets) #use the transform you developed with train data to get test features

    predictions = classifier.predict(matrix) # use the classifier predictions for the test features
    return predictions

In [30]:
# Fill in best classifier in your function and re-trian your classifier using all training data
# Get predictions for unlabelled test data
# 2% credits
classifier = learn_classifier(X, y, best_penalty)
unlabeled_tweets = pd.read_csv("test.csv", na_filter=False)
y_pred = classify_tweets(tfidf, classifier, unlabeled_tweets)

predictions_df = pd.DataFrame(y_pred, columns=["y_pred"])  # 转换为 DataFrame
predictions_df.to_csv("labels.csv", index=False)  # 保存为 CSV 文件

Did your Logistic Regression classifier perform better than the baseline (while evaluating with training data)? Explain in 1-2 sentences how you reached this conclusion.

*YOUR ANSWER HERE*

### l2 penalty performs better with respect to average (across folds) cross validation accuracy by approximately 25%. 

# Let's evaluate with the test labels (see extra credit homework on gradescope -- I will provide the test labels after extra credit homework due date)

In [29]:
test_file = pd.read_csv("test-w-labels.csv")
test_labels = create_labels(test_file) 

FileNotFoundError: [Errno 2] No such file or directory: 'test-w-labels.csv'

In [ ]:
test_accuracy = accuracy_score(test_labels, y_pred)
print(test_accuracy)


0.7537688442211056


In [ ]:
#baseline test accuracy with either 0 or 1 label for all the tweets
print(max(sum(test_labels)/len(test_labels),1-sum(test_labels)/len(test_labels)))

0.5577889447236181


## As in the case of average cross validation accuracy, the logistic classifier achieves about 20% more accuracy compared to majority label classifier (yay!)